In [1]:
# Tecnología
import json
import calendar
import pandas as pd
from sparky_bc import Sparky
import datetime as dt
from dateutil.relativedelta import relativedelta

# files lz conection
path_sparky_conf = '/Users/santlond/Documents/sparky_conf.json'

# Configurar conexión a LZ
with open(path_sparky_conf, 'rb') as JSON_lz_File:
    sp_config = json.loads(JSON_lz_File.read())
    
USER='santlond'
PASS=sp_config['ID']
DSN='IMPALA_PROD'
LOGDIR= 'logs'
# sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp")
sparky = Sparky(username=USER, password=PASS, dsn=DSN, hostname="sbmdeblze004.bancolombia.corp", spark_submit="spark3-submit")
# sparky = Sparky(username=USER, password=PASS, dsn=DSN)

helper = sparky.helper

/Users/santlond/Documents/venv_py39_odbc/lib/python3.9/site-packages/helper/helper.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-12-23 09:17:02 - [WARNING] - No se encontro la carpeta "/Users/santlond/Documents/ADQUIRENCIA_FERIA_EVA/logs" para guardar los logs


 ____  _____ __  __  ___ _____ _____ 
|  _ \| ____|  \/  |/ _ \_   _| ____|
| |_) |  _| | |\/| | | | || | |  _|  
|  _ <| |___| |  | | |_| || | | |___ 
|_| \_\_____|_|  |_|\___/ |_| |_____|
                                     
 ____  ____   _    ____  _  __
/ ___||  _ \ / \  |  _ \| |/ /
\___ \| |_) / _ \ | |_) | ' / 
 ___) |  __/ ___ \|  _ <| . \ 
|____/|_| /_/   \_\_| \_\_|\_\
                              



## Uso o activo

Un comercio usa adquirencia en un mes específico o mes de análisis cuando:

- Métrica Normal: tienen al menos 1 trx aporbada en ese mes.

- Métrica 5x: tiene al menos 1 trx aprobada en el transucurso de un año, incluyendo el mes de análisis


### Análisis ingestión compras tabla transaccional

In [2]:
# Se deben esperar dos días para que se ingeste la información completa de las transacciones de un día en particular
# Para obtener la información de las compras del día jueves y viernes se debe espear hasta la sgte semana
sql = """
WITH outcome1 AS
  (SELECT YEAR,
          MONTH,
          DAY,
          left(cast(f_trx as string), 10) as f_trx,
          count(*) AS num_compras
   FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
   WHERE YEAR IN (2025)
     AND MONTH BETWEEN 11 AND 12
     AND DAY BETWEEN 1 AND 31
     and tipo_trx = "Purchase"
   GROUP BY 1,
            2,
            3,
            4
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC),
     outcome2 AS
  (SELECT *,
          sum(num_compras) OVER (PARTITION BY f_trx) AS total_compras,
                                sum(num_compras) OVER (PARTITION BY f_trx
                                                       ORDER BY YEAR,
                                                                MONTH,
                                                                DAY ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS num_compras_cumsum
   FROM outcome1
   ORDER BY f_trx DESC, YEAR DESC, MONTH DESC, DAY DESC)
SELECT *,
       row_number() OVER (PARTITION BY f_trx
                          ORDER BY YEAR,
                                   MONTH,
                                   DAY) AS num_ing,
                         round(num_compras /total_compras, 4) AS prop,
                         round(num_compras_cumsum / total_compras, 4) AS prop_cumsum
FROM outcome2
ORDER BY f_trx DESC,
         YEAR DESC, MONTH DESC, DAY DESC;
"""
df_prueba = helper.obtener_dataframe(sql)

2025-12-23 09:18:00 - [INFO] - Transcurrido: 1766499481, Tiempo de Refresco = 1000


------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 1/1 DATAFRAME        descargando   09:18:00 AM             

2025-12-23 09:19:07 - [INFO] - 1,442 filas, 10 columnas, 01:06.4 consultando, 00:00.5 descargando, 00:00.0 convirtiendo


 1/1 DATAFRAME         finalizado   09:18:00 AM     01:07.2 
------------------------------------------------------------


In [3]:
df_prueba.head(40)

,year,month,day,f_trx,num_compras,total_compras,num_compras_cumsum,num_ing,prop,prop_cumsum
0,2025,12,11,2026-01-03,1,1,1,1,1.0000,1.0000
1,2025,12,10,2026-01-02,1,1,1,1,1.0000,1.0000
2,2025,12,22,2025-12-28,1,2,2,2,0.5000,1.0000
3,2025,12,17,2025-12-28,1,2,1,1,0.5000,0.5000
4,2025,12,4,2025-12-24,1,1,1,1,1.0000,1.0000
5,2025,12,22,2025-12-22,58,58,58,1,1.0000,1.0000
6,2025,12,22,2025-12-21,2783559,2783559,2783559,1,1.0000,1.0000
7,2025,12,22,2025-12-20,3259757,3259757,3259757,1,1.0000,1.0000
8,2025,12,22,2025-12-19,2678383,2678445,2678445,3,1.0000,1.0000
9,2025,12,19,2025-12-19,61,2678445,62,2,0.0000,0.0000


## Construcción histórico transacciones

In [5]:
# Verificar cantidad de registros por partición
# La última ingestión usada 2025-12-11 para obtener las transacciones. # MODIFICAR
sql = """
SELECT YEAR,
       mes,
       dia,
       count(*) AS frec
FROM proceso_vdm.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2022 AND 2025
  AND mes BETWEEN 1 AND 12
  AND dia BETWEEN 1 AND 31
GROUP BY 1,
         2,
         3
ORDER BY YEAR DESC, mes DESC,
                    dia DESC;
"""
df_prueba = helper.obtener_dataframe(sql)


------------------------------------------------------------
  i    tipo    nombre    estado     hora_inicio   duracion   
------------------------------------------------------------
 3/3 DATAFRAME              error   09:23:02 AM     00:00.5 
------------------------------------------------------------

------------------------------------------------------------
Consulta que fallo:

SELECT YEAR,
       mes,
       dia,
       count(*) AS frec
FROM proceso_vdm.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2022 AND 2025
  AND mes BETWEEN 1 AND 12
  AND dia BETWEEN 1 AND 31
GROUP BY 1,
         2,
         3
ORDER BY YEAR DESC, mes DESC,
                    dia DESC;

------------------------------------------------------------

------------------------------------------------------------
[HY000] [Cloudera][ImpalaODBC] (370) Query analysis error occurred during query execution: [HY000] : AnalysisException: Could not resolve table reference: 'proceso_vdm.mdo_adquirencia_trxs'
 (370) (SQLExecD

Error: ('HY000', "[HY000] [Cloudera][ImpalaODBC] (370) Query analysis error occurred during query execution: [HY000] : AnalysisException: Could not resolve table reference: 'proceso_vdm.mdo_adquirencia_trxs'\n (370) (SQLExecDirectW)")

In [61]:
df_prueba.head(20)

,year,mes,dia,frec
0,2025,11,14,71303
1,2025,11,13,70841
2,2025,11,12,70689
3,2025,11,11,70064
4,2025,11,10,187826
5,2025,11,7,72516
6,2025,11,6,71600
7,2025,11,5,70348
8,2025,11,4,237328
9,2025,10,31,74259


In [62]:
# Construcción histórico trxs
# Se selecciona el rango de tiempo de las ingestiones a almacenar [Esto para efectos de facilitar la actualización del histórico]
fecha_inicial = '2025-11-15' # MODIFICAR. DEBE SER EL PRIMER DÍA DE INGESTIÓN DE TRANSACCIONES A ALMACENAR O EL SIGUIENTE DÍA DESPUÉS DEL ÚLTIMO EN UNA ACTUALIZACIÓN.
fecha_final = '2025-12-11' # MODIFICAR. DEBE SER EL ÚLTIMO DÍA DE INGESTIÓN DE TRANSACCIONES ALMACENADAS O EL DÍA MÁS RECIENTE EN UNA ACTUALIZACIÓN

fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='D')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config

,fechas,year,month,day
0,2025-11-15,2025,11,15
1,2025-11-16,2025,11,16
2,2025-11-17,2025,11,17
3,2025-11-18,2025,11,18
4,2025-11-19,2025,11,19
5,2025-11-20,2025,11,20
6,2025-11-21,2025,11,21
7,2025-11-22,2025,11,22
8,2025-11-23,2025,11,23
9,2025-11-24,2025,11,24


In [ ]:
# # Crear tabla que almacenará la información

# sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_trxs;"""
# helper.ejecutar_consulta(sql_drop)

# sql = """
# CREATE TABLE proceso_vdm.mdo_adquirencia_trxs  (
#                 cod_unico VARCHAR,
#                 f_trx STRING,
#                 num_trxs BIGINT,
#                 mnt_total_trxs DECIMAL(38,2),
#                 DIA INT
#                 )
#             PARTITIONED BY 
#             (
#             YEAR INT,
#             MES INT
#             )
# STORED AS PARQUET
# TBLPROPERTIES ('transactional' = 'false');
# """
# helper.ejecutar_consulta(sql)

# sql_compute = """COMPUTE STATS proceso_vdm.mdo_adquirencia_trxs;"""

2025-12-02 16:18:06 - [INFO] - Transcurrido: 1764710286, Tiempo de Refresco = 1000


----------------------------------------------------------------------------------
  i  tipo              nombre                 estado     hora_inicio   duracion   
----------------------------------------------------------------------------------
 1/1 DROP proceso_vdm.mdo_adquirencia_trxs   finalizado   04:18:06 PM     00:00.3 
----------------------------------------------------------------------------------
------------------------------------------------------------------------------------
  i   tipo               nombre                 estado     hora_inicio   duracion   
------------------------------------------------------------------------------------
 2/2 CREATE proceso_vdm.mdo_adquirencia_trxs   finalizado   04:18:07 PM     00:00.2 
------------------------------------------------------------------------------------


In [63]:
# Iterar para obtener las trxs por cliente
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Exrayendo datos de las particiones: ', str(row.year), '-', str(row.month), '-', str(row.day))
    print('')
    print('Obteniendo transacciones adquirencia de los comercios')
    print('')

    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)

    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    SELECT cod_unico,
       to_date(f_trx) as f_trx,
       count(*) AS num_trxs,
       sum(mnt_total_trx) AS mnt_total_trxs,
        """ + str(row.day) + """ AS DIA,
        """ + str(row.year) + """ AS YEAR,
        """ + str(row.month) + """ AS MES
    FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
    WHERE YEAR = """ + str(row.year) + """
     AND MONTH = """ + str(row.month) + """
     AND DAY = """ + str(row.day) + """
     AND LOWER(TRIM(tipo_trx)) = "purchase"
     AND LOWER(TRIM(estado_trx)) = "cleared"
    GROUP BY 1,
            2;"""
    helper.ejecutar_consulta(sql)

    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando transacciones adquirencia de los comercios')
    print('')

    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_trxs PARTITION (YEAR, MES)
    SELECT cod_unico,
           f_trx,
           num_trxs,
           mnt_total_trxs,
           DIA,
           YEAR,
           MES
    FROM proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql)
    print('')

##################################################

Exrayendo datos de las particiones:  2025 - 11 - 15

Obteniendo transacciones adquirencia de los comercios

-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 92/92      DROP        proceso.mdo_adquirencia_trxs_temp   finalizado   11:02:08 PM     00:00.3 
-------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------
   i     tipo                     nombre                     estado     hora_inicio   duracion   
-------------------------------------------------------------------------------------------------
 93/93    CREATE        proceso.mdo_adquirencia_trxs_tem

In [64]:
# Verficar cantidad de registros desde la tabla fuente
sql = """
with outcome as (
SELECT cod_unico,
       to_date(f_trx) as f_trx,
       count(*) AS num_trxs,
       sum(mnt_total_trx) AS mnt_total_trxs
    FROM resultados_vspc_medios_de_pago.gsap_m_transaccional
    WHERE YEAR = 2022
     AND MONTH = 1
     AND DAY = 1
     AND LOWER(TRIM(tipo_trx)) = "purchase"
     AND LOWER(TRIM(estado_trx)) = "cleared"
    GROUP BY 1,
            2
            )
SELECT count(*)
FROM outcome;
"""
helper.obtener_dataframe(sql)

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 200/200 DATAFRAME                                           descargando   11:06:47 PM             

2025-12-11 23:06:48 - [INFO] - 1 filas, 1 columnas, 00:01.0 consultando, 00:00.3 descargando, 00:00.0 convirtiendo


 200/200 DATAFRAME                                            finalizado   11:06:47 PM     00:01.5 
---------------------------------------------------------------------------------------------------


,count(*)
0,64516


In [65]:
# Verificar cantidad de registros por partición
# La última ingestión usada 2025-11-14 para obtener las transacciones. # MODIFICAR FECHA
sql = """
SELECT YEAR,
       mes,
       dia,
       count(*) AS frec
FROM proceso_vdm.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2022 AND 2025
  AND mes BETWEEN 1 AND 12
  AND dia BETWEEN 1 AND 31
GROUP BY 1,
         2,
         3
ORDER BY YEAR DESC, mes DESC, dia DESC;
"""
helper.obtener_dataframe(sql).head(20)

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 201/201 DATAFRAME                                           descargando   11:07:04 PM             

2025-12-11 23:07:07 - [INFO] - 965 filas, 4 columnas, 00:02.7 consultando, 00:00.2 descargando, 00:00.0 convirtiendo


 201/201 DATAFRAME                                            finalizado   11:07:04 PM     00:03.3 
---------------------------------------------------------------------------------------------------


,year,mes,dia,frec
0,2025,12,11,76464
1,2025,12,10,73743
2,2025,12,9,245384
3,2025,12,5,76202
4,2025,12,4,75369
5,2025,12,3,75015
6,2025,12,2,75215
7,2025,12,1,197904
8,2025,11,28,75372
9,2025,11,27,74232


## Construcción histórico transacciones por mes

In [66]:
# Crear tabla que almacenará las transacciones menusales por cliente
# EN ACTUALIZACIÓN SE VULEVE A CREAR LA TABLA DESDE CERO
sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_trxs_mes_1 PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso_vdm.mdo_adquirencia_trxs_mes_1 (
                cod_unico VARCHAR,
                num_trxs BIGINT,
                mnt_total_trxs DECIMAL(38,2),
                periodo_trxs INT
            )
STORED AS PARQUET
TBLPROPERTIES ('transactional' = 'false');
"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE STATS proceso_vdm.mdo_adquirencia_trxs_mes_1;"""
helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 202/202      DROP   proceso_vdm.mdo_adquirencia_trxs_mes_1   finalizado   11:07:23 PM     00:00.4 
---------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 203/203    CREATE   proceso_vdm.mdo_adquirencia_trxs_mes_1   finalizado   11:07:24 PM     00:00.4 
---------------------------------------------------------------------------------------------------


In [45]:
# # Construcción histórico trxs mes
# # Se selecciona el rango de tiempo de las ingestiones a almacenar [Esto para efectos de facilitar la actualización del histórico]
# fecha_inicial = '2022-01-01'
# fecha_final = '2025-11-10' # MODIFICAR. DEBE SER EL ÚLTIMO DÍA DE INGESTIÓN DE TRANSACCIONES ALMACENADAS O EL DÍA MÁS RECIENTE EN UNA ACTUALIZACIÓN
# fecha_inicial_ts = pd.to_datetime(fecha_inicial)
# fecha_final_ts = pd.to_datetime(fecha_final)
# fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='YS')

# # # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# # pri_dia_part = fechas[-1] + relativedelta(days=1)
# # pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# # ult_dia_part = fechas[-1] + relativedelta(days=10)
# # ult_dia_part = ult_dia_part.date().isoformat()
# # fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# # fechas = fechas.append(fechas_faltantes)

# # df_config
# df_config = pd.DataFrame({'fechas': fechas})
# df_config['year_inicial'] = df_config['fechas'].dt.year
# df_config['month'] = df_config['fechas'].dt.month
# df_config['day'] = df_config['fechas'].dt.day
# df_config['fechas_fin_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1) - relativedelta(days=1))
# df_config['year_fin_mes'] = df_config['fechas_fin_mes'].dt.year
# df_config['month_fin_mes'] = df_config['fechas_fin_mes'].dt.month
# df_config['day_fin_mes'] = df_config['fechas_fin_mes'].dt.day

# # Transformar fechas_fin_mes como el último día con ingestión para facilitar el momento de la actualización sin generar duplicados
# df_config.loc[df_config.shape[0] - 1, 'fechas_fin_mes'] = fecha_final_ts

# df_config

In [68]:
print('#' * 50)
print('')
print('Exrayendo datos de las particiones: ', str(row.year), '-', str(row.month))
print('')
print('Obteniendo transacciones adquirencia de los comercios')
print('')

sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_mes_1_temp PURGE;"""
helper.ejecutar_consulta(sql_drop)

sql = """
CREATE TABLE proceso.mdo_adquirencia_trxs_mes_1_temp STORED AS PARQUET AS
SELECT cod_unico,
       cast(replace(left(cast(f_trx AS string), 7), '-', '') AS INT) AS periodo_trxs,
       count(*) AS num_trxs,
       sum(mnt_total_trxs) AS mnt_total_trxs
FROM proceso_vdm.mdo_adquirencia_trxs
WHERE YEAR BETWEEN 2022 AND """ + str(df_config.year.values[-1]) + """
AND MES BETWEEN 1 AND 12
GROUP BY 1,
       2;"""
helper.ejecutar_consulta(sql)

sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_trxs_mes_1_temp;"""
helper.ejecutar_consulta(sql_compute)

print('')
print('Insertando transacciones adquirencia de los comercios')
print('')

sql = """
INSERT INTO proceso_vdm.mdo_adquirencia_trxs_mes_1
SELECT cod_unico,
       num_trxs,
       mnt_total_trxs,
       periodo_trxs
FROM proceso.mdo_adquirencia_trxs_mes_1_temp;"""
helper.ejecutar_consulta(sql)
print('')

sql_compute = """COMPUTE STATS proceso_vdm.mdo_adquirencia_trxs_mes_1;"""
helper.ejecutar_consulta(sql_compute)

##################################################

Exrayendo datos de las particiones:  2025 - 12

Obteniendo transacciones adquirencia de los comercios

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 205/205      DROP  proceso.mdo_adquirencia_trxs_mes_1_temp   finalizado   11:07:36 PM     00:00.7 
---------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 206/206    CREATE  proceso.mdo_adquirencia_t

## Construcción históricos

Se calcula la métrica uso para tres escenarios

- Todos los clientes
- Clientes nuevos
- Clientes viejos

Se dice que un cliente [todos, nuevos o viejos] tiene uso cuando realiza al menos una (1) transaccion en el año corriente. 

Importante tener en cuenta para clientes nuevos. Un cliente que vinculó adquirencia en el año 2024 [nuevo en 2024] y realizó un transacción en el mismo año suma a la métrica, en cambio si ese mismo cliente realiza una transacción en el año 2025 y siguiente no sumará a la métrica.

In [ ]:
# # Tabla que almacenará comercios con trxs por periodo
# sql_drop = """DROP TABLE IF EXISTS proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist PURGE;"""
# helper.ejecutar_consulta(sql_drop)

# sql = """
# CREATE TABLE proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist  (
#                 codigo_unico DOUBLE,
#                 periodo DOUBLE,
#                 num_trxs BIGINT,
#                 mnt_total_trxs DECIMAL(38,2),
#                 tipo_cliente STRING,
#                 producto STRING
#                 )
# STORED AS PARQUET
# TBLPROPERTIES ('transactional' = 'false');
# """
# helper.ejecutar_consulta(sql)

# sql_compute = """COMPUTE STATS proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist;"""
# helper.ejecutar_consulta(sql_compute)

---------------------------------------------------------------------------------------------
  i   tipo                    nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------
 9/9    DROP ...a_y_wompi_vinculaciones_con_trxs_hist   finalizado   04:34:42 PM     00:00.2 
---------------------------------------------------------------------------------------------
-----------------------------------------------------------------------------------------------
   i    tipo                    nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------
 10/10  CREATE ...a_y_wompi_vinculaciones_con_trxs_hist   finalizado   04:34:43 PM     00:00.2 
-----------------------------------------------------------------------------------------------
--------------------------------------------------

In [73]:
# Construcción histórico trxs
# Se selecciona el rango de tiempo de las transacciones a extraer
fecha_inicial = '2025-11-01' # MODIFICAR. INICIO DE UN MES. EN ACTUALIZACIÓN USAR EL SIGUIENTE MES DESPUÉS DEL ÚLTIMO ALMACENADO
fecha_final = '2025-11-30' # MODIFICAR. FIN DE UN MES. EN ACTUALIZACIÓN USAR EL MES RECIENTE CON TRANSACCIONES COMPLETAS
fecha_inicial_ts = pd.to_datetime(fecha_inicial)
fecha_final_ts = pd.to_datetime(fecha_final)
fechas = pd.date_range(start=fecha_inicial, end=fecha_final, freq='M')

# # Ir por las particiones del siguiente mes que contendrá las compras de los últimos días del reango de tiempo deseado
# pri_dia_part = fechas[-1] + relativedelta(days=1)
# pri_dia_part = pri_dia_part.date().isoformat() # Primer día de partición para buscar compras del primer día.
# ult_dia_part = fechas[-1] + relativedelta(days=10)
# ult_dia_part = ult_dia_part.date().isoformat()
# fechas_faltantes = pd.date_range(start=pri_dia_part, end=ult_dia_part, freq='D')

# # siguiente_dias = dt.datetime(siguientes_dias.year, siguientes_dias.month, calendar.monthrange(siguientes_dias.year, siguientes_dias.month)[1])
# fechas = fechas.append(fechas_faltantes)

# df_config
df_config = pd.DataFrame({'fechas': fechas})
df_config['year'] = df_config['fechas'].dt.year
df_config['month'] = df_config['fechas'].dt.month
df_config['day'] = df_config['fechas'].dt.day
df_config['periodo'] = df_config['year'] * 100 + df_config['month']
df_config['periodo_base_year'] = df_config['year'] * 100 + 1
df_config['periodo_sgte_mes'] = df_config['fechas'].apply(lambda x: x + relativedelta(months=1))
df_config['year_sgte_mes'] = df_config['periodo_sgte_mes'].dt.year
df_config['periodo_sgte_mes'] = df_config['periodo_sgte_mes'].apply(lambda x: x.year * 100 + x.month)
df_config['periodo_viejos'] = df_config['fechas'].apply(lambda x: str(x.year - 1) + '12')
# df_config['ult_6_meses_fin'] = df_config['fechas'].apply(lambda x: x - relativedelta(months=5))
# df_config['ult_6_meses_inicio'] = df_config['ult_6_meses_fin'].apply(lambda x: x.replace(day=1))
# df_config['year_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.year
# df_config['month_ult_6_meses'] = df_config['ult_6_meses_fin'].dt.month
# df_config['periodo_ult_6_meses'] = df_config['year_ult_6_meses'] * 100 + df_config['month_ult_6_meses']
df_config

,fechas,year,month,day,periodo,periodo_base_year,periodo_sgte_mes,year_sgte_mes,periodo_viejos
0,2025-11-30,2025,11,30,202511,202501,202512,2025,202412


In [74]:
for row in df_config.itertuples():
    print('#' * 50)
    print('')
    print('Obteniendo Métrica 5X')
    print('')
    print('Mes de Análisis: ', str(row.year), '-', str(row.month))
    print('')

    print('Viejos')
    print('') 
    print('Obteniendo vinculaciones del último mes del año anterior correspondiente al Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT codigo_unico, min(periodo) AS periodo
       FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
       WHERE YEAR <= """ + str(row.year_sgte_mes) + """
       AND MONTH BETWEEN 1 AND 12
       AND DAY BETWEEN 1 AND 31
       AND periodo <= """ + row.periodo_viejos + """
       GROUP BY 1
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM vinc;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'viejos' as tipo_cliente,
         'adqui' as producto
    FROM proceso_vdm.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente,
           b.producto
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')


    print('Nuevos')
    print('') 
    print('Obteniendo vinculaciones del año correspondiente acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp_1 PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp_1 STORED AS PARQUET AS
    WITH vinc AS (
       SELECT codigo_unico, min(periodo) AS periodo
       FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
       WHERE YEAR BETWEEN """ + str(row.year) + """ AND """ + str(row.year_sgte_mes) + """
       AND MONTH BETWEEN 1 AND 12
       AND DAY BETWEEN 1 AND 31
       AND periodo BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
       GROUP BY 1
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM vinc;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_vinculaciones_temp_1;"""
    helper.ejecutar_consulta(sql_compute)
    print('')
    print('Eliminar los nuevos que ya aparecen en los viejos, se puede presentar múltiples razones, entre ellas: adición de franquicias, la adición de una nueva franquicia genera un nuevo registro en la tabla')
    print('')
    print('Crear tabla con los viejos')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_viejos_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_viejos_temp STORED AS PARQUET AS
    SELECT codigo_unico, periodo
    FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
    WHERE tipo_cliente = 'viejos'"""
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_vinculaciones_viejos_temp;"""
    helper.ejecutar_consulta(sql_compute)
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
    SELECT a.codigo_unico, a.periodo
    FROM proceso.mdo_adquirencia_vinculaciones_temp_1 AS a
    LEFT ANTI JOIN proceso.mdo_adquirencia_vinculaciones_viejos_temp AS b on a.codigo_unico = b.codigo_unico
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'nuevos' as tipo_cliente,
         'adqui' as producto
    FROM proceso_vdm.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente,
           b.producto
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')

    print('Todos')
    print('') 
    print('Obteniendo vinculaciones acumuladas hasta el Mes de análisis')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_vinculaciones_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_vinculaciones_temp STORED AS PARQUET AS
    WITH vinc AS (
       SELECT codigo_unico, min(periodo) AS periodo
       FROM resultados_vspc_medios_de_pago.gsap_cun_conso_cu_vf
       WHERE YEAR <= """ + str(row.year_sgte_mes) + """
       AND MONTH BETWEEN 1 AND 12
       AND DAY BETWEEN 1 AND 31
       AND periodo <= """ + str(row.periodo) + """
       GROUP BY 1
    )
    SELECT codigo_unico, """ + str(row.periodo) + """ AS periodo
    FROM vinc;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_vinculaciones_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Obtener las transacciones año base corrido')
    print('')
    sql_drop = """DROP TABLE IF EXISTS proceso.mdo_adquirencia_trxs_temp PURGE;"""
    helper.ejecutar_consulta(sql_drop)
    sql = """
    CREATE TABLE proceso.mdo_adquirencia_trxs_temp STORED AS PARQUET AS
    select cod_unico,
         sum(num_trxs) as num_trxs,
         sum(mnt_total_trxs) as mnt_total_trxs,
         'todos' as tipo_cliente,
         'adqui' as producto
    FROM proceso_vdm.mdo_adquirencia_trxs_mes_1
    WHERE periodo_trxs BETWEEN """ + str(row.periodo_base_year) + """ AND """ + str(row.periodo) + """
    GROUP BY 1;
    """
    helper.ejecutar_consulta(sql)
    sql_compute = """COMPUTE STATS proceso.mdo_adquirencia_trxs_temp;"""
    helper.ejecutar_consulta(sql_compute)

    print('')
    print('Insertando en tabla proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist')
    print('')
    
    sql = """
    INSERT INTO proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
    SELECT a.codigo_unico,
           a.periodo,
           b.num_trxs,
           b.mnt_total_trxs,
           b.tipo_cliente,
           b.producto
    FROM proceso.mdo_adquirencia_vinculaciones_temp AS a
    INNER JOIN proceso.mdo_adquirencia_trxs_temp AS b ON CAST(a.codigo_unico AS BIGINT) = CAST(b.cod_unico AS BIGINT);
    """
    helper.ejecutar_consulta(sql)
    print('')

    

##################################################

Obteniendo Métrica 5X

Mes de Análisis:  2025 - 11

Viejos

Obteniendo vinculaciones del último mes del año anterior correspondiente al Mes de análisis

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 235/235      DROP ...so.mdo_adquirencia_vinculaciones_temp   finalizado   11:16:17 PM     00:00.4 
---------------------------------------------------------------------------------------------------
---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
-----------------------------------------------------------------------------------------------

In [75]:
# Obtener uso por mes
# Número de Vinculaciones
sql = """
SELECT producto,
       tipo_cliente,
       periodo,
       count(*) AS num_vinc_uso_cumsum_ym
FROM proceso_vdm.mdo_adquirencia_y_wompi_vinculaciones_con_trxs_hist
GROUP BY 1,
         2,
         3
ORDER BY periodo DESC, tipo_cliente, producto;
"""
df_prueba = helper.obtener_dataframe(sql)

---------------------------------------------------------------------------------------------------
    i      tipo                     nombre                     estado     hora_inicio   duracion   
---------------------------------------------------------------------------------------------------
 262/262 DATAFRAME                                           descargando   11:18:08 PM             

2025-12-11 23:18:14 - [INFO] - 279 filas, 4 columnas, 00:05.1 consultando, 00:00.4 descargando, 00:00.0 convirtiendo


 262/262 DATAFRAME                                            finalizado   11:18:08 PM     00:05.7 
---------------------------------------------------------------------------------------------------


In [76]:
df_prueba.head(20)

,producto,tipo_cliente,periodo,num_vinc_uso_cumsum_ym
0,adqui,nuevos,202511.0,23445
1,adqui,todos,202511.0,102281
2,adqui,viejos,202511.0,78836
3,adqui,nuevos,202510.0,21390
4,wompi,nuevos,202510.0,11275
5,adqui,todos,202510.0,99996
6,wompi,todos,202510.0,28226
7,adqui,viejos,202510.0,78606
8,wompi,viejos,202510.0,16951
9,adqui,nuevos,202509.0,19020
